# 02 — LIB, CORPUS, and VOCAB Tables
Builds the three core tables of the digital analytical edition.

**Inputs:** `data/docs_sampled.csv`

**Outputs:** `data/hc3_LIB.csv`, `data/hc3_CORPUS.csv`, `data/hc3_VOCAB.csv`

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

ps = PorterStemmer()
stop_words = set(stopwords.words('english'))
print('Libraries loaded.')

Libraries loaded.


In [2]:
docs_sampled = pd.read_csv('../data/docs_sampled.csv')
print(f'Loaded {len(docs_sampled):,} documents')

Loaded 10,258 documents


## LIB Table

In [3]:
LIB = docs_sampled[['doc_id','question_id','source','author_type','char_length','word_count']].copy()
LIB = LIB.set_index('doc_id')
LIB.index.name = 'doc_id'

print(f'LIB shape: {LIB.shape}')
print(f'Avg char length: {LIB["char_length"].mean():.1f}')
print(f'Columns: {list(LIB.columns)}')
LIB.head()

LIB shape: (10258, 5)
Avg char length: 796.1
Columns: ['question_id', 'source', 'author_type', 'char_length', 'word_count']


,question_id,source,author_type,char_length,word_count
doc_id,,,,,
22649_AI0,22649,finance,chatgpt,1315,207
20814_AI0,20814,finance,chatgpt,2148,390
22988_AI0,22988,finance,chatgpt,1110,180
22547_AI1,22547,finance,chatgpt,1837,283
21591_AI0,21591,finance,chatgpt,1207,195


In [4]:
LIB.to_csv('../data/hc3_LIB.csv')
print('Saved data/hc3_LIB.csv')

Saved data/hc3_LIB.csv


In [5]:
print('Document count by source and author type:')
print(LIB.groupby(['source', 'author_type']).size().unstack().to_string())

Document count by source and author type:
author_type  chatgpt  human
source                     
finance          541    472
medicine         160    150
open_qa          426    143
reddit_eli5     2000   6164
wiki_csai        101    101


## CORPUS Table

In [6]:
def get_pos_group(tag):
    if tag.startswith('NN'): return 'NOUN'
    if tag.startswith('VB'): return 'VERB'
    if tag.startswith('JJ'): return 'ADJ'
    if tag.startswith('RB'): return 'ADV'
    return 'OTHER'

corpus_records = []
for _, row in docs_sampled.iterrows():
    paragraphs = [p.strip() for p in str(row['text']).split('\n') if p.strip()]
    for para_num, para in enumerate(paragraphs):
        sentences = sent_tokenize(para)
        for sent_num, sent in enumerate(sentences):
            tokens = word_tokenize(sent)
            tags   = nltk.pos_tag(tokens)
            for token_num, (token, tag) in enumerate(tags):
                corpus_records.append({
                    'source': row['source'],
                    'question_id': row['question_id'],
                    'doc_id': row['doc_id'],
                    'author_type': row['author_type'],
                    'para_num': para_num,
                    'sent_num': sent_num,
                    'token_num': token_num,
                    'token_str': token,
                    'term_str': token.lower(),
                    'pos': tag,
                    'pos_group': get_pos_group(tag)
                })

CORPUS = pd.DataFrame(corpus_records)
CORPUS= CORPUS.set_index(['source','question_id','doc_id','para_num','sent_num','token_num'])
print(f'CORPUS rows: {len(CORPUS):,}')
CORPUS.head(10)

CORPUS rows: 1,607,435


author_type  \
source  question_id doc_id    para_num sent_num token_num               
finance 22649       22649_AI0 0        0        0             chatgpt   
                                                1             chatgpt   
                                                2             chatgpt   
                                                3             chatgpt   
                                                4             chatgpt   
                                                5             chatgpt   
                                                6             chatgpt   
                                                7             chatgpt   
                                                8             chatgpt   
                                                9             chatgpt   

                                                          token_str  term_str  \
source  question_id doc_id    para_num sent_num token_num                       
finance 22649       22649_AI0 0        0        0               Yes       yes   
                                                1                 ,         ,   
                                                2                it        it   
                                                3                is        is   
                                                4          possible  possible   
                                                5               for       for   
                                                6               you       you   
                                                7                as        as   
                                                8                 a         a   
                                                9          computer  computer   

                                                           pos pos_group  
source  question_id doc_id    para_num sent_num token_num                 
finance 22649       22649_AI0 0        0        0           UH     OTHER  
                                                1            ,     OTHER  
                                                2          PRP     OTHER  
                                                3          VBZ      VERB  
                                                4           JJ       ADJ  
                                                5           IN     OTHER  
                                                6          PRP     OTHER  
                                                7           IN     OTHER  
                                                8           DT     OTHER  
                                                9           NN      NOUN

In [7]:
CORPUS.to_csv('../data/hc3_CORPUS.csv')
print('Saved data/hc3_CORPUS.csv')

Saved data/hc3_CORPUS.csv


## VOCAB Table

In [8]:
CORPUS_reset = CORPUS.reset_index()

term_counts = CORPUS_reset.groupby('term_str').size().rename('n')
doc_counts= CORPUS_reset.groupby('term_str')['doc_id'].nunique().rename('df')

VOCAB = pd.concat([term_counts, doc_counts], axis=1)
VOCAB['p'] = VOCAB['n'] / VOCAB['n'].sum()
VOCAB['i']= -np.log2(VOCAB['p'])
VOCAB['dfidf'] = VOCAB['df'] * VOCAB['i']
VOCAB['porter_stem'] = VOCAB.index.map(lambda x: ps.stem(x))
VOCAB['stop']= VOCAB.index.map(lambda x: x in stop_words)

pos_mode = CORPUS_reset.groupby('term_str')['pos'].agg(lambda x: x.value_counts().index[0])
pos_group_mode = CORPUS_reset.groupby('term_str')['pos_group'].agg(lambda x: x.value_counts().index[0])
VOCAB['max_pos'] = pos_mode
VOCAB['max_pos_group'] = pos_group_mode
VOCAB.index.name= 'term_str'

VOCAB = VOCAB.sort_values('dfidf', ascending=False)
print(f'VOCAB size: {len(VOCAB):,}')
print('\nTop 20 significant words by DFIDF (non-stop):')
print(VOCAB[~VOCAB['stop']].head(20)[['n','df','dfidf','max_pos_group']].to_string())

VOCAB size: 40,531

Top 20 significant words by DFIDF (non-stop):
               n    df         dfidf max_pos_group
term_str                                          
.          65061  9961  46087.789811         OTHER
,          66268  9000  41402.737901         OTHER
's          9834  4540  33381.559137          VERB
)           8620  3716  28029.255927         OTHER
(           8326  3680  27941.949746         OTHER
n't         5270  3002  24774.730653           ADV
also        3857  2840  24716.706131           ADV
like        3979  2511  21740.586492         OTHER
one         3522  2336  20636.573876         OTHER
people      4570  2204  18642.204545          NOUN
:           4995  2149  17901.298955         OTHER
``          8358  2330  17678.611538         OTHER
make        2871  1914  17472.889452          VERB
time        2985  1876  17020.598602          NOUN
would       3345  1881  16756.960870         OTHER
way         2423  1715  16075.981731          NOUN
get         2662

In [9]:
VOCAB.to_csv('../data/hc3_VOCAB.csv')
print('Saved data/hc3_VOCAB.csv')

Saved data/hc3_VOCAB.csv
